In [ ]:
%pip install gdown pandas numpy matplotlib seaborn scipy statsmodels
!gdown https://drive.google.com/drive/folders/1oZtAvuySav-Z8mYWGNPj_wsno2Vb3zp2?usp=sharing

Using cookies from /root/.cache/gdown/cookies.txt
Retrieving folder contents
Processing file 1OX2OvSAMXoUvtafv70WAfN2AVQ3JuP-b Behavior of the urban traffic of the city of Sao Paulo in Brazil.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1OX2OvSAMXoUvtafv70WAfN2AVQ3JuP-b
To: /content/Dados/Behavior of the urban traffic of the city of Sao Paulo in Brazil.csv
100% 5.73k/5.73k [00:00<00:00, 9.50MB/s]
Download completed


# Versões de Bibliotecas
As versões usadas durante a construção e execução desse *notebook* são:
- gdown : 6.5.2
- pandas : 3.1.1
- numpy : 2.5.5
- matplotlib : 3.11.2
- seaborn : 0.13.2
- scipy : 1.18.1
- statsmodels : 0.15.0

As versões instaladas pelo comando acima podem ser verificadas usando o comando (como na célula abaixo):
- *pip show gdown pandas numpy matplotlib seaborn scipy statsmodels | grep -E "^(Name|Version):"*

In [2]:
!pip show gdown pandas numpy matplotlib seaborn scipy statsmodels | grep -E "^(Name|Version):"

Name: gdown
Version: 5.2.2
Name: pandas
Version: 2.2.3
Name: numpy
Version: 2.4.6
Name: matplotlib
Version: 3.10.0
Name: seaborn
Version: 0.13.2
Name: scipy
Version: 1.16.3
Name: statsmodels
Version: 0.15.0


# Preparação dos dados
Primeiro deve ser feita a importação das bibliotecas e dos dados brutos em formato csv, depois o arquivo bruto será transformado em um *Pandas Dataset*.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
arquivo_dados = "/content/Dados/Behavior of the urban traffic of the city of Sao Paulo in Brazil.csv" #<- note que a pasta "content" se refere ao sistema de arquivos do collab
#arquivo_dados = "/Dados/Behavior of the urban traffic of the city of Sao Paulo in Brazil.csv" #<- a versão offline deve omitir a pasta content
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
raw_data = pd.read_csv(arquivo_dados, sep=";", decimal=",", encoding="latin-1")
data = raw_data.copy()
data.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Dados/Behavior of the urban traffic of the city of Sao Paulo in Brazil.csv'

Com os dados importados podem ser feito um pré-processamento, em específico podemos registrar em variáveis (a fim de facilitar consultas futuras):
- Nomes das colunas que descrevem os acidentes ficaram armazenados na lista *incidentes*;
- A coluna *"Slowness in traffic (%)"* ficou armazenada na variável *slowness*;
- A coluna *"Hour (Coded)"* ficou armazenada na variável *hour*;

Além disso podemos calcular para cada registro os seguintes dados:
- O dia em que foi coletado (*day_coded*);
- Se teve acidente ou não no intervalo analisado (*has_incident*);
- Qual o total de incidentes no intervalo analisado (*incident_total*);

In [ ]:
incidentes = [col for col in data.columns if col not in ["Slowness in traffic (%)", "Hour (Coded)"]]
slowness = "Slowness in traffic (%)"
hour = "Hour (Coded)"
data["day_coded"] = np.arange(len(data)) / 27
data['day_coded'] = data['day_coded'].astype(int)
data['Hour (Coded)'] = data['Hour (Coded)'].astype(int)
data['has_incident'] = (data[incidentes].sum(axis=1) > 0).astype(int)
data['incident_total'] = data[incidentes].sum(axis=1)
data.head()

# Análise Exploratória
Podemos começar por uma análise superficial, verificando quais colunas possuem valores ausentes e quais são as dimensões do *dataset*.

In [ ]:

print(f"Ausentes:\n{data.isna().sum()}")
print(f"\nDimensões: {data.shape}\n")


Como não há dados faltantes, podemos avançar para estatísticas descritivas da variável lentidão, a fim de entender o seu comportamento.

In [ ]:
slowness_describe = data[slowness].describe()
slowness_describe["variance"] = data[slowness].var()
slowness_describe["iqr"] = data[slowness].quantile(0.75) - data[slowness].quantile(0.25)
slowness_describe

Auxiliarmente, podemos analisar a distribuição da lentidão usando um histograma e a sua dispersão e presença de *outliers* usando um *boxplot*.

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(12,4))

sns.histplot(
    data=data,
    x=slowness,
    ax=axs[0]
)

sns.boxplot(
    data=data,
    y=slowness,
    ax=axs[1]
)

axs[0].set_title("Distribuição da Lentidão")
axs[0].set_xlabel("Lentidão Percentual")
axs[0].set_ylabel("Total")
axs[1].set_title("Boxplot da Lentidão")
axs[1].set_ylabel("Lentidão Percentual")
plt.show()

Abaixo ficam identificados os registros que se encaixam como *outliers*. Note que esses *outliers* não devem ser descartados uma vez que podem estar relacionados tanto à ocorrência de incidentes ou ao período (dia ou hora) em que o intervalo foi coletado.

In [ ]:
data[data[slowness]>18.5]

Sendo assim analisar a lentidão por dia e hora podem revelar tendências interessantes.

In [ ]:
plt.figure(figsize=(12,4))
slow_por_dia = data.groupby('day_coded')[slowness].agg(["count", "mean", "std"])
plt.errorbar(
    x=slow_por_dia.index,
    y=slow_por_dia['mean'],
    yerr=slow_por_dia['std']
)
plt.xticks(slow_por_dia.index)
plt.title("Lentidão por dia")
plt.xlabel("Dia (Código)")
plt.ylabel("Lentidão Média Percentual")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
slow_por_hora = data.groupby(hour)[slowness].agg(["count", "mean", "std"])
plt.errorbar(
    x=slow_por_hora.index,
    y=slow_por_hora['mean'],
    yerr=slow_por_hora['std']
)
plt.xticks(slow_por_hora.index)
plt.title("Lentidão por hora")
plt.xlabel("Hora (Código)")
plt.ylabel("Lentidão Média Percentual")
plt.grid(True)
plt.show()

Tendo em vista as variaçoes de lentidão por dia e por hora, podemos correlacionalas utilizando uma mapa de calor, mostrando os pontos da semana que concentram a maior parte da lentidão.

In [ ]:
plt.figure(figsize=(16,4))
sns.heatmap(
    data=data.pivot_table(
        index="day_coded",
        columns=hour,
        values=slowness,
        aggfunc="mean"
    ),
    cmap="YlOrRd",
    annot=True,
    cbar_kws={"label": "Lentdião Média Percentual"}
)
plt.xlabel("Hora (Código)")
plt.ylabel("Dia (Código)")
plt.show()

As análises acima nos servem para relacionar a lentidão ao períodos (dia e hora) analisados, partiremos agora para uma análise focada na presença e ausência de acidentes. Começaremos por uma análise de boxplot para averiguar as distribuição e presença de *outliers* em cada grupo.

In [ ]:
sns.boxplot(
    data=data,
    x="has_incident",
    y=slowness,
    hue="has_incident",
    legend=False
)
plt.xticks([0,1],["Sem Incidente", "Com Incidente"])
plt.title("Lentidão com e sem incidentes")
plt.ylabel("Lentidão Percentual")
plt.xlabel("")
plt.show()

*Outliers* do grupo com acidentes. Aqui há uma divisão entre outliers com muitos acidentes (mais que 4) e poucos acidentes (4 ou menos), logo ainda não podemos afirmar se os acidentes aumentam a lentidão ou não.

In [ ]:
data[(data[slowness]>18.5) & (data['has_incident'] == 1)]

*Outliers* do grupo sem acidentes. Aqui a presença desses *outliers* parece ser explicada pela alta lentidão observada em horários tardios (entre às 18:00 e 20:00) dos dias 2 (terça-feira) e 3 (quarta-feira).

In [ ]:
data[(data[slowness]>15) & (data['has_incident'] == 0)]

Como anteriormente podemos investigar a variação nos níveis de lentidão considerando dia e hora para cada grupo, a fim de averiguar como o intervalo observado é afetado pelas variáveis temporais.

In [ ]:
slipted_data = data.copy()
slipted_data["incident_id"] = data["has_incident"].map({0: "Sem Incidente", 1: "Com Incidente"})
plt.figure(figsize=(12,4))
sns.lineplot(
    data=slipted_data,
    x=hour,
    y=slowness,
    hue="incident_id",
    estimator="mean",
    err_style='bars'
)
plt.xticks(range(1,28))
plt.xlabel("Horas (Código)")
plt.ylabel("Lentidão Media Percentual")
plt.legend(
    title="Categoria",
    loc="upper left"
)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
sns.lineplot(
    data=slipted_data,
    x="day_coded",
    y=slowness,
    hue="incident_id",
    err_style='bars'
)
plt.xticks(range(0,5))
plt.ylabel("Lentidão Percentual")
plt.xlabel("Dias (Código)")
plt.legend(
    title="Categoria",
    loc="upper left"
)
plt.grid(True)
plt.show()

# Estátistica Inferencial

In [ ]:
from scipy.stats import mannwhitneyu
statistic, p_value = mannwhitneyu(
    x=data.loc[data["has_incident"] == 0, slowness],
    y=data.loc[data["has_incident"] == 1, slowness],
    alternative="two-sided"
)

print(f"Estatística U:{statistic}")
print(f"P-value:{p_value}")

In [ ]:
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
resultados = []
for incidente in incidentes:
  rho, p_value = spearmanr(
    a=data[incidente],
    b=data[slowness]
  )
  resultados.append({
      "incidente": incidente,
      "Rho": rho,
      "P-Valor" : p_value
  })

resultados_ajustados = pd.DataFrame(resultados)
resultados_ajustados["P-Ajustado"] = multipletests(
    pvals=resultados_ajustados["P-Valor"],
    method="fdr_bh"
)[1]
resultados_ajustados.sort_values(by="P-Ajustado")

In [ ]:
import statsmodels.formula.api as smf
resultados = []
for incidente in incidentes:
  modelo = smf.ols(
      formula=f'Q("{slowness}") ~ Q("{incidente}") + C(day_coded) + C(Q("Hour (Coded)"))',
      data=data
  ).fit(cov_type="HC3")
  resultados.append({
      "Incidente": incidente,
      "P-Valor": modelo.pvalues[f'Q("{incidente}")']
  })

resultados_ajustados = pd.DataFrame(resultados)
resultados_ajustados["P-Ajustado"] = multipletests(
    pvals=resultados_ajustados["P-Valor"],
    method="fdr_bh"
)[1]
resultados_ajustados.sort_values(by="P-Ajustado")
